In [1]:
import kagglehub
path = kagglehub.dataset_download("andrewmvd/cyberbullying-classification")

100%|██████████| 2.82M/2.82M [00:00<00:00, 171MB/s]

Extracting files...


In [2]:
!pip install langdetect
!pip install emoji
!pip install nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993335 sha256=9015442b4048a2bfaea97a8758def2be81ca6464c2aaed8bc93116b9d425ce8c
  Stored in directory: /root/.cache/pip/wheels/eb/87/25/2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 11.5 MB/s eta 0:00:00


In [3]:
import pandas as pd
path = path + "/cyberbullying_tweets.csv"

In [4]:
# import Check dataset
df = pd.read_csv(path)
print(df.shape)
print(df.columns)
df.head()

(47692, 2)
Index(['tweet_text', 'cyberbullying_type'], dtype='object')


,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying


In [5]:
# Find unique values in the 'category' column
categories = df['cyberbullying_type'].unique()
print(categories)

['not_cyberbullying' 'gender' 'religion' 'other_cyberbullying' 'age'
 'ethnicity']


# Pre- processing

In [6]:
#lowercase conversion
df['tweet_text'] = df['tweet_text'].str.lower()


In [7]:
import emoji

# Find tweets containing at least one emoji
emoji_tweets = df[df['tweet_text'].apply(emoji.emoji_count) > 0]

emoji_tweets.head()

,tweet_text,cyberbullying_type
21,kids love😘❤ @ mohamad bin zayed city مدينة محم...,not_cyberbullying
80,@imrbarfield it starts thursday. my classes ar...,not_cyberbullying
135,mmmm #mkr forget deconstruction @lisamromano @...,not_cyberbullying
144,rt @mykitchenrules: nawwww 😭😭😭 #mkr,not_cyberbullying
151,@justinbieber your how i get through bullying ...,not_cyberbullying


In [8]:
#emoji to text conversion
df['tweet_text'] = df['tweet_text'].apply(emoji.demojize)
print(df.loc[21])
df.head()

tweet_text            kids love:face_blowing_a_kiss::red_heart: @ mo...
cyberbullying_type                                    not_cyberbullying
Name: 21, dtype: object


,tweet_text,cyberbullying_type
0,"in other words #katandandre, your food was cra...",not_cyberbullying
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying


In [9]:
# Remove missing/null tweets
df = df.dropna(subset=['tweet_text'])

# Convert to string
df['tweet_text'] = df['tweet_text'].astype(str)

# Remove empty or whitespace-only tweets
df = df[df['tweet_text'].str.strip() != '']

# Remove duplicate tweets
df = df.drop_duplicates(subset=['tweet_text'])

# Reset index
df = df.reset_index(drop=True)

print("Remaining tweets:", len(df))

Remaining tweets: 45991


In [10]:
import re

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove @mentions
    text = re.sub(r'@\w+', '', text)

    # Remove special characters and numbers and all non-ASCII characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()


    return text

# Apply cleaning
df['clean_tweet'] = df['tweet_text'].apply(clean_text)
df.head()

,tweet_text,cyberbullying_type,clean_tweet
0,"in other words #katandandre, your food was cra...",not_cyberbullying,in other words katandandre your food was crapi...
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,why is aussietv so white mkr theblock imaceleb...
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,a classy whore or more red velvet cupcakes
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks for the heads up but not too conc...
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,this is an isis account pretending to be a kur...


In [11]:
# Remove tweets that are extremely short
df = df[df['clean_tweet'].str.len() >= 3]
print("Remaining tweets:", len(df))

Remaining tweets: 45701


In [12]:
# Detect the language
from langdetect import detect, LangDetectException

def detect_language(text):
    try:
        return detect(str(text))
    except LangDetectException:
        return 'unknown'

df['language'] = df['clean_tweet'].apply(detect_language)
# See which tweets are NOT English
non_english = df[df['language'] != 'en']
print(non_english[['clean_tweet', 'language']])

                                             clean_tweet language
6      itu sekolah ya bukan tempat bully ga jauh kaya...       id
12                                            ughhhh mkr       sw
15     parem de fazer bullying comigo uhahuah bando d...       pt
16              tadinhu de mim sofrendo bulling viu mimi       pt
39                                              fuck mkr       da
...                                                  ...      ...
45944  look how long we had google niggers still dumb...       af
45946                       lmao nigga dont you got work       tl
45950  im dark skin you dumb fuck rt you need to die ...       af
45953                 fuck obama dumb ass nigger mad ehh       et
45973                      fuck you doris ya dumb nigger       id

[3056 rows x 2 columns]


In [13]:
print(non_english['cyberbullying_type'].value_counts())

cyberbullying_type
not_cyberbullying      1389
other_cyberbullying     716
ethnicity               517
gender                  384
religion                 28
age                      22
Name: count, dtype: int64


In [14]:
# Remove non-English tweets
df = df[df['language'] == 'en'].copy()

# Remove the temporary language column
df.drop(columns=['language'], inplace=True)

print("Remaining tweets:", len(df))

Remaining tweets: 42645


In [15]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [16]:
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords')

# English stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):

    # Remove stopwords
    text = ' '.join(
        word for word in text.split()
        if word not in stop_words
    )

    return text

# Apply cleaning
df['clean_tweet'] = df['clean_tweet'].apply(clean_text)
df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,tweet_text,cyberbullying_type,clean_tweet
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...


In [17]:
# Apply lemmatization
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

df['lemmatized_tweet'] = df['clean_tweet'].apply(lemmatize_text)

df[['clean_tweet', 'lemmatized_tweet']].head()

,clean_tweet,lemmatized_tweet
0,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr
1,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...
2,classy whore red velvet cupcakes,classy whore red velvet cupcake
3,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...
4,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...


VADER was selected because it is lightweight, interpretable, designed for short informal social-media text, and provides continuous sentiment features that can be incorporated alongside TF-IDF features without requiring computationally expensive language models.

In [18]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.6 MB/s eta 0:00:00


In [19]:
import nltk
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

analyzer = SentimentIntensityAnalyzer()

# Calculate VADER sentiment scores
df['sentiment_scores'] = df['tweet_text'].apply(
    analyzer.polarity_scores
)

# Extract four sentiment scores
df['negative'] = df['sentiment_scores'].apply(lambda x: x['neg'])
df['positive'] = df['sentiment_scores'].apply(lambda x: x['pos'])
df['neutral']  = df['sentiment_scores'].apply(lambda x: x['neu'])
df['compound'] = df['sentiment_scores'].apply(lambda x: x['compound'])

df[['lemmatized_tweet','negative','positive','neutral','compound']].head()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


,lemmatized_tweet,negative,positive,neutral,compound
0,word katandandre food crapilicious mkr,0.000,0.000,1.000,0.0000
1,aussietv white mkr theblock imacelebrityau tod...,0.000,0.000,1.000,0.0000
2,classy whore red velvet cupcake,0.320,0.199,0.481,-0.4137
3,meh p thanks head concerned another angry dude...,0.243,0.150,0.607,-0.4854
4,isi account pretending kurdish account like is...,0.129,0.180,0.691,0.0258


In [20]:
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258


In [21]:
df_cleaned = df

df_cleaned.to_csv('df_cleaned.csv', index=False)

print(f"Saved {len(df_cleaned)} rows to df_cleaned.csv")

Saved 42645 rows to df_cleaned.csv
